<a href="https://colab.research.google.com/github/TAUforPython/hybrid-ODE-reservoir-PhysioNet/blob/master/heart-model-animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
"""
Heart Model ODE Integration.ipynb
Revised script integrating the ODE-based heart model into the original visualization framework.
"""

# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Polygon
from IPython.display import display, clear_output
import ipywidgets as widgets
from IPython.core.display import HTML
from functools import lru_cache
from scipy.integrate import odeint
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.rcParams.update({
    'font.size': 9,
    'axes.linewidth': 1.2,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'axes.titlesize': 11,
    'axes.labelsize': 9
})

# Fixed seed for reproducibility
RANDOM_SEED = 37
np.random.seed(RANDOM_SEED)


# Case Hemodinamics modeling

In [27]:
# -*- coding: utf-8 -*-
"""
Heart Model ODE Integration.ipynb
Revised script integrating the ODE-based heart model into the original visualization framework.
Includes enhanced hemodynamics explanation figure.
"""

# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Polygon
from IPython.display import display, clear_output
import ipywidgets as widgets
from IPython.core.display import HTML
from functools import lru_cache
import pandas as pd
import warnings
from scipy.integrate import odeint # Added for ODE solving
warnings.filterwarnings('ignore')

# Set up plotting style
plt.rcParams.update({
    'font.size': 9,
    'axes.linewidth': 1.2,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'axes.titlesize': 11,
    'axes.labelsize': 9
})

# Fixed seed for reproducibility
RANDOM_SEED = 37
np.random.seed(RANDOM_SEED)

# --- 2. BASE MODEL PARAMETERS (Updated for ODE Model) ---
# These will be used by
base_params_ode = {
    # Elastances (mmHg/ml) - Values adapted for ODE model
    'Emax': 2.0,  # Maximum elastance (contractility)
    'Emin': 0.06, # Minimum elastance (diastolic stiffness)

    # Resistances (mmHg*s/ml)
    'Rp': 1.0,    # Pulmonary resistance
    'Ra': 1.0,    # Aortic resistance
    'Rin': 0.01,  # Mitral/tricususp. resistance

    # Capacitances (ml/mmHg)
    'Ca': 1.0,    # Arterial compliance
    'Cv': 10.0,   # Venous compliance

    # Volumes (ml)
    'Vd': 5.0,    # Unstressed LV volume (V0_lv analog)

    # Time parameters (s)
    'T_heart': 0.8,  # Heart cycle period (T)
}

# Global parameters for the ODE model (will be set inside the simulation)
T = base_params_ode['T_heart']
Tsys = 0.3 * np.sqrt(T)
Tir = 0.5 * Tsys

# --- 3. HELPER FUNCTIONS (for ODE integration) ---
def init_glob_para(HC):
    """Initialize global parameters for the ODE model."""
    global T, Tsys, Tir
    T = HC
    Tsys = 0.3 * np.sqrt(T)
    Tir = 0.5 * Tsys

def Esin(t):
    """Activation function for elastance."""
    heart_cycle = int(t / T)
    t_norm = t - heart_cycle * T
    if (Tsys + Tir < t_norm) and (t_norm < T):
        return 0
    elif (Tsys < t_norm) and (t_norm < Tsys + Tir):
        return 0.5 * (1 + np.cos(np.pi * ((t_norm - Tsys) / Tir)))
    elif (0 < t_norm) and (t_norm < Tsys):
        return 0.5 * (1 - np.cos(np.pi * (t_norm / Tsys)))
    else:
        return 0

def Elastance(Emax, Emin, t):
    """Instantaneous elastance."""
    return Esin(t) * (Emax - Emin) + Emin

def Plv(Vlv, Vd, Emax, Emin, t):
    """LV pressure based on elastance and volume."""
    return Elastance(Emax, Emin, t) * (Vlv - Vd)

def Qa(Plv, Pa, Ra):
    """Aortic flow (positive when exiting LV)."""
    if Plv > Pa:
        return (Plv - Pa) / Ra
    else:
        return 0.0

def Qin(Plv, Pv, Rin):
    """Inflow to LV (positive when filling)."""
    if Pv > Plv:
        return (Pv - Plv) / Rin
    else:
        return 0.0

def Qp(Pa, Pv, Rp):
    """Pulmonary flow."""
    return (Pa - Pv) / Rp

def heart_ode(y, t, Rp, Ra, Rin, Ca, Cv, Vd, Emax, Emin):
    """The core ODE system."""
    Vlv, Pa, Pv = y
    Plv_val = Plv(Vlv, Vd, Emax, Emin, t)
    Qin_val = Qin(Plv_val, Pv, Rin)
    Qa_val = Qa(Plv_val, Pa, Ra)
    Qp_val = Qp(Pa, Pv, Rp)
    dVlv_dt = Qin_val - Qa_val
    dPa_dt = (Qa_val - Qp_val) / Ca
    dPv_dt = (Qp_val - Qin_val) / Cv
    return [dVlv_dt, dPa_dt, dPv_dt]

def compute_ode(Rp, Ra, Rin, Ca, Cv, Vd, Emax, Emin, t, start_v, start_pa, start_pv):
    """Solve the ODE system."""
    y0 = [start_v, start_pa, start_pv]
    sol = odeint(heart_ode, y0, t, args=(Rp, Ra, Rin, Ca, Cv, Vd, Emax, Emin))
    result_Vlv = sol[:, 0]
    result_Pa = sol[:, 1]
    result_Pv = sol[:, 2]
    return (result_Vlv, result_Pa, result_Pv)

def validate_params(params):
    """Validate physiological correctness of parameters."""
    checks = [
        (params['Emax'] > params['Emin'], "Emax must be > Emin"),
        (params['Rp'] > 0, "Rp must be > 0"),
        (params['Ra'] > 0, "Ra must be > 0"),
        (params['Rin'] > 0, "Rin must be > 0"),
        (params['Ca'] > 0, "Ca must be > 0"),
        (params['Cv'] > 0, "Cv must be > 0"),
        (params['T_heart'] <= 2.0 and params['T_heart'] >= 0.4, "Heart period T_heart must be 0.4–2.0 s"),
    ]
    for ok, msg in checks:
        if not ok:
            raise ValueError(f"❌ Invalid parameter: {msg}")
    return True

def generate_ecg(t, heart_rate=75, noise_std=0.02, seed=None):
    """Generate ECG trace."""
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng(RANDOM_SEED)
    T_period = 60.0 / heart_rate
    ecg = np.zeros_like(t)
    for i, ti in enumerate(t):
        tc = (ti % T_period) / T_period # Normalize within cycle
        if 0.05 <= tc <= 0.25:
            ecg[i] = 0.15 * np.sin(np.pi * (tc - 0.05) / 0.2)
        elif 0.25 <= tc <= 0.35:
            ecg[i] = 1.0 - 1.5 * abs((tc - 0.3) / 0.1) # QRS complex
        elif 0.4 <= tc <= 0.6:
            ecg[i] = 0.3 * np.sin(np.pi * (tc - 0.4) / 0.2) # T wave
        ecg[i] += rng.normal(0, noise_std)
    return ecg

# --- 4. CORE SIMULATION (Replaced with ODE Solver) ---
def _simulate_core(params, duration, fs):
    """Core simulation using the ODE model."""
    t = np.linspace(0, duration, int(duration * fs))
    dt = t[1] - t[0] if len(t) > 1 else 0.002

    # Update global ODE parameters
    init_glob_para(params['T_heart'])

    # Initial conditions for ODE
    V_lv_init = params.get('V0_lv', 120.0) + 10 # Start slightly above Vd
    Pa_init = 90.0 # Approximate aortic pressure
    Pv_init = 5.0  # Approximate venous pressure

    # Solve ODE
    V_lv, P_ao, P_v = compute_ode(
        params['Rp'], params['Ra'], params['Rin'],
        params['Ca'], params['Cv'], params['Vd'],
        params['Emax'], params['Emin'],
        t, V_lv_init, Pa_init, Pv_init
    )

    # Calculate LV pressure from solved volume and elastance
    P_lv = np.array([Plv(V_lv[i], params['Vd'], params['Emax'], params['Emin'], t[i]) for i in range(len(t))])

    # Calculate derived pressures for visualization (approximations)
    P_ra = P_la = P_v # Simplified approximation for atrial pressures
    P_rv = P_v # Simplified approximation for RV pressure

    # Calculate elastance over time for potential display
    E_lv = np.array([Elastance(params['Emax'], params['Emin'], ti) for ti in t])

    # Generate ECG trace
    hr = 60 / params['T_heart']
    ecg = generate_ecg(t, heart_rate=hr, noise_std=0.015, seed=RANDOM_SEED)

    # Return arrays matching original function signature
    # Note: V_rv, flow_mitral, flow_aortic are placeholders for compatibility
    V_rv = np.full_like(V_lv, 100.0) # Placeholder
    flow_mitral = np.full_like(V_lv, 0.0) # Placeholder
    flow_aortic = np.full_like(V_lv, 0.0) # Placeholder

    return t, V_lv, P_lv, V_rv, P_rv, P_ao, E_lv, ecg, flow_mitral, flow_aortic

def simulate_heart(params, duration=2.4, fs=250):
    """Wrapper for simulation with validation."""
    validate_params(params)
    fs = min(fs, 250) # Limit for Colab performance
    return _simulate_core(params, duration, fs)

# --- 5. VISUALIZATION AND OTHER FUNCTIONS (Remain mostly unchanged) ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
# Assuming 'find_peaks' is available, otherwise you can implement a simple peak finding logic

# --- Функция для отрисовки обновленной PV-петли ---
def plot_pv_loop(ax, V_lv, P_lv, t, params):
    """Plot the PV loop calculated from simulation results, with corrected ES/ED and enhanced visualization."""
    ax.clear()
    ax.plot(V_lv, P_lv, 'r-', linewidth=2.5, label='PV-Loop LV', zorder=3)
    ax.fill_between(V_lv, P_lv, alpha=0.1, color='red', zorder=1) # More standard fill

    # Find approximate indices for ES and ED using derivatives or specific features
    dt = t[1] - t[0] if len(t) > 1 else 0.002
    cycle_length = int(params['T_heart'] / dt)
    num_cycles = int(len(t) / cycle_length) if cycle_length > 0 else 1
    if num_cycles == 0: num_cycles = 1

    # Analyze the last *full* cardiac cycle for ES/ED
    start_of_last_cycle = (num_cycles - 1) * cycle_length
    if start_of_last_cycle < 0: start_of_last_cycle = 0
    end_of_last_cycle = min(num_cycles * cycle_length, len(V_lv))

    V_lv_last_cycle = V_lv[start_of_last_cycle:end_of_last_cycle]
    P_lv_last_cycle = P_lv[start_of_last_cycle:end_of_last_cycle]
    t_last_cycle = t[start_of_last_cycle:end_of_last_cycle]

    if len(V_lv_last_cycle) > 0:
        # Method 1: Find peaks in P_lv (systolic peaks) and troughs in V_lv (diastolic troughs) within the cycle
        # This often works well for simulated data
        p_lv_peaks, _ = find_peaks(P_lv_last_cycle)
        v_lv_troughs, _ = find_peaks(-V_lv_last_cycle) # Invert for troughs

        # Method 2: Alternative - Find min/max within approximated phases (less robust for irregular shapes)
        # start_systole_approx = int(0.2 * len(P_lv_last_cycle))
        # end_systole_approx = int(0.7 * len(P_lv_last_cycle))
        # es_idx_rel = np.argmin(V_lv_last_cycle[start_systole_approx:end_systole_approx]) + start_systole_approx
        # ed_idx_rel = np.argmax(V_lv_last_cycle[end_systole_approx:]) + end_systole_approx

        # Prioritize Method 1 (peaks/troughs) if found, otherwise fall back to min/max
        if len(p_lv_peaks) > 0 and len(v_lv_troughs) > 0:
            # Assume the highest P_lv peak corresponds to end-systole pressure
            # and the lowest V_lv trough corresponds to end-diastole volume
            # This might need refinement depending on the exact shape
            # Let's refine: ES is minimum volume during ejection, ED is max volume during filling.
            # After P_lv peak, volume decreases -> ES is the minimum V_lv near the P_lv peak.
            # Before P_lv peak, volume increases -> ED is the maximum V_lv before the P_lv peak.

            # Find the most prominent systolic peak (highest pressure in cycle)
            sys_peak_idx_rel = p_lv_peaks[np.argmax(P_lv_last_cycle[p_lv_peaks])]

            # ES: Find the minimum volume around and just after the systolic pressure peak
            # Define a window around the peak where ejection occurs (volume decreases)
            window_start = max(0, sys_peak_idx_rel - 5) # Adjust window size if needed
            window_end = min(len(V_lv_last_cycle), sys_peak_idx_rel + 10) # Adjust window size if needed
            es_idx_rel_in_window = np.argmin(V_lv_last_cycle[window_start:window_end]) + window_start

            # ED: Find the maximum volume just before the systolic pressure peak
            # Define a window before the peak where filling occurs (volume increases)
            window_start_ed = max(0, sys_peak_idx_rel - 15) # Adjust window size if needed
            window_end_ed = sys_peak_idx_rel # Up to the peak
            ed_idx_rel_in_window = np.argmax(V_lv_last_cycle[window_start_ed:window_end_ed]) + window_start_ed

            # Validate indices within the window
            if 0 <= es_idx_rel_in_window < len(V_lv_last_cycle):
                 es_idx = start_of_last_cycle + es_idx_rel_in_window
            else:
                 es_idx = start_of_last_cycle + np.argmin(V_lv_last_cycle) # Fallback

            if 0 <= ed_idx_rel_in_window < len(V_lv_last_cycle):
                 ed_idx = start_of_last_cycle + ed_idx_rel_in_window
            else:
                 ed_idx = start_of_last_cycle + np.argmax(V_lv_last_cycle) # Fallback

        else: # Fallback if no clear peaks/troughs found
            # Use simple min/max volume approach on the last cycle
            es_idx_rel = np.argmin(V_lv_last_cycle)
            ed_idx_rel = np.argmax(V_lv_last_cycle)
            es_idx = start_of_last_cycle + es_idx_rel
            ed_idx = start_of_last_cycle + ed_idx_rel

        # --- Mark ES and ED ---
        ESV = V_lv[es_idx]
        EDV = V_lv[ed_idx]
        P_es = P_lv[es_idx]

        ax.scatter([ESV], [P_es], c='red', s=80, zorder=5, edgecolors='black', linewidth=2, label='ES')
        ax.scatter([EDV], [P_lv[ed_idx]], c='blue', s=80, zorder=5, edgecolors='black', linewidth=2, label='ED')
        ax.text(ESV + (max(V_lv) - min(V_lv)) * 0.02, P_es, 'ES', ha='left', fontsize=10, fontweight='bold', color='red')
        ax.text(EDV + (max(V_lv) - min(V_lv)) * 0.02, P_lv[ed_idx], 'ED', ha='left', fontsize=10, fontweight='bold', color='blue')
        # ---

        # --- Calculate Emax, Ea, SW, EF ---
        if ESV > params.get('V0_lv', params['Vd']) and EDV > ESV: # Use V0_lv if defined, otherwise Vd
            V0_lv = params.get('V0_lv', params['Vd'])
            Emax = P_es / (ESV - V0_lv) # End-systolic Elastance
            Ea = P_es / (EDV - ESV)    # Effective Arterial Elastance
            SV = EDV - ESV             # Stroke Volume
            EF = (SV / EDV) * 100      # Ejection Fraction (%)

            # --- Draw ESPVR line (Emax) ---
            ax.plot([V0_lv, ESV], [0, P_es], '--', color='orange', linewidth=2, label=f'ESPVR (Emax = {Emax:.2f})')

            # --- Draw EDPVR line (approximate) ---
            # A common approximation is a line from origin to ED point, or using a curvilinear fit.
            # For simplicity, drawing a line from V0_lv to EDV at a typical diastolic pressure gradient.
            # A better approximation might use the initial part of the diastolic filling.
            # Here, we draw a line from V0_lv to EDV, assuming a very steep slope initially.
            # Let's use the pressure at EDV and connect to a point representing V0_lv at a lower pressure (e.g., 0 or 5 mmHg).
            # Or, use the actual simulated P_lv at EDV.
            # P_ed = P_lv[ed_idx] # Usually close to 5-10 mmHg baseline, let's use simulated value
            # A simple linear EDPVR from V0_lv to EDV might not be physiologically accurate.
            # We'll represent it less prominently.
            # ax.plot([V0_lv, EDV], [5, P_ed], ':', color='green', linewidth=1, label=f'EDPVR (approx)') # Less common to draw linearly
            # Instead, we can just label the EDV and ED pressure implicitly.

            # --- Stroke Work (SW) Calculation (Area inside the loop) ---
            # Use the last cycle for accuracy
            dV_cycle = np.diff(V_lv_last_cycle)
            P_avg_cycle = (P_lv_last_cycle[:-1] + P_lv_last_cycle[1:]) / 2
            SW = np.trapz(P_avg_cycle * dV_cycle) # mmHg * ml

            # --- Ventricular-Arterial Coupling ---
            coupling = Emax / Ea if Ea > 0 else float('inf')

            # --- Efficiency Metric (Optional, e.g., Stroke Work normalized) ---
            # Sometimes shown as SW/EDV or SW/ESV or related metrics.
            # SW_normalized = SW / EDV # Example metric

            # --- Add Text Box with Metrics ---
            metrics_text = (
                f'ESV: {ESV:.1f} ml\n'
                f'EDV: {EDV:.1f} ml\n'
                f'SV: {SV:.1f} ml\n'
                f'EF: {EF:.1f}%\n'
                f'Emax: {Emax:.2f} mmHg/ml\n'
                f'Ea: {Ea:.2f} mmHg/ml\n'
                f'Emax/Ea: {coupling:.2f}\n'
                f'SW: {SW:.0f} mmHg*ml'
            )
            ax.text(0.02, 0.98, metrics_text, transform=ax.transAxes,
                    fontsize=8, verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))

            # --- Coupling Status Indicator (like in previous versions) ---
            if 1.0 <= coupling <= 2.5:
                eff_status = 'Optimal'
                eff_color = 'lightgreen'
            elif coupling < 1.0:
                eff_status = 'Underload'
                eff_color = 'lightcoral'
            else:
                eff_status = 'Hyperdynamic'
                eff_color = 'lightyellow'
            ax.text(0.02, 0.02, f'Coupling: {eff_status}', transform=ax.transAxes,
                    fontsize=9, verticalalignment='bottom', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=eff_color, alpha=0.8))

        else:
             ax.text(0.02, 0.98, 'Calculation Error (ESV<=V0 or EDV<=ESV)', transform=ax.transAxes,
                     fontsize=9, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

    else:
        ax.text(0.02, 0.98, 'Insufficient data for ES/ED calc', transform=ax.transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

    ax.set_xlabel('Volume LV (ml)', fontweight='bold')
    ax.set_ylabel('Pressure LV (mmHg)', fontweight='bold')
    ax.set_title('PV-Loop Left Ventricle (Corrected ES/ED & Metrics)')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='lower right', fontsize=8)
    # Optional: Set consistent limits if needed for comparison across runs
    # ax.set_xlim(left=min(V_lv)*0.95, right=max(V_lv)*1.05)
    # ax.set_ylim(bottom=min(P_lv)*0.95, top=max(P_lv)*1.05)






def draw_heart(ax, params, t_cur, V_lv, P_lv, V_rv, P_rv, P_ao, E_lv):
    """Draw the anatomical heart diagram."""
    ax.clear()
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('🫀 Model Heart (ODE Simulation)', fontsize=12, fontweight='bold', pad=10)

    arterial = '#FF6B6B'
    venous = '#4ECDC4'

    # Calculate dynamic sizes based on simulation state (simplified)
    # Use average values or a proxy from the ODE results
    lv_size_factor = 0.8 + 0.2 * np.sin(t_cur * 2 * np.pi / params['T_heart']) # Proxy for contraction
    la_size_factor = 0.9 + 0.1 * np.cos(t_cur * 2 * np.pi / params['T_heart']) # Proxy for filling

    # LA
    la_x, la_y, la_w, la_h = 3, 8, 1.2 * la_size_factor, 1.0
    la = Rectangle((la_x - la_w/2, la_y), la_w, la_h, facecolor=arterial, alpha=0.7, edgecolor='darkred', linewidth=2)
    ax.add_patch(la)
    ax.text(la_x, la_y + 0.5, 'LA', ha='center', va='center', fontsize=10, fontweight='bold')

    # LV
    lv_x, lv_y = 3, 5
    lv_w = 1.8 * lv_size_factor
    lv_h = 3.0 * lv_size_factor
    lv_center = (lv_x, lv_y - lv_h/2)
    lv = Circle(lv_center, lv_w/2, facecolor=arterial, alpha=0.8, edgecolor='darkred', linewidth=2)
    ax.add_patch(lv)
    ax.text(lv_x, lv_y - lv_h/2, 'LV', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(lv_x, lv_y - lv_h - 0.5, f'P={P_lv:.1f} mmHg', ha='center', fontsize=7)

    # RA
    ra_x, ra_y = 7, 8
    ra_w = 1.2 * la_size_factor
    ra = Rectangle((ra_x - ra_w/2, ra_y), ra_w, 1.0, facecolor=venous, alpha=0.7, edgecolor='darkblue', linewidth=2)
    ax.add_patch(ra)
    ax.text(ra_x, ra_y + 0.5, 'RA', ha='center', va='center', fontsize=10, fontweight='bold')

    # RV
    rv_x, rv_y = 7, 5
    rv_w = 1.5 * lv_size_factor
    rv_h = 2.5 * lv_size_factor
    rv_points = [(rv_x - rv_w/2, rv_y), (rv_x + rv_w/2, rv_y),
                 (rv_x + rv_w/3, rv_y - rv_h), (rv_x - rv_w/3, rv_y - rv_h)]
    rv = Polygon(rv_points, facecolor=venous, alpha=0.8, edgecolor='darkblue', linewidth=2)
    ax.add_patch(rv)
    ax.text(rv_x, rv_y - rv_h/2, 'RV', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(rv_x, rv_y - rv_h - 0.5, f'P={P_rv:.1f}', ha='center', fontsize=7)

    # Ao
    ao_x, ao_y = 3, 10.5
    ao_w, ao_h = 0.8, 1.5
    ao = Rectangle((ao_x - ao_w/2, ao_y), ao_w, ao_h, facecolor=arterial, alpha=0.9, edgecolor='darkred', linewidth=2)
    ax.add_patch(ao)
    ax.text(ao_x, ao_y + ao_h/2, 'Ao', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(ao_x, ao_y + ao_h + 0.3, f'P={P_ao:.1f}', ha='center', fontsize=7)

    # Valves and vessels (simplified arrows)
    ax.annotate('', xy=(la_x, la_y), xytext=(lv_x, lv_y), arrowprops=dict(arrowstyle='-|>', color='k', lw=1.5, alpha=0.7))
    ax.text(4.2, 6.5, 'MK', fontsize=7, color='k', alpha=0.7)
    ax.annotate('', xy=(lv_x, lv_y), xytext=(ao_x, ao_y), arrowprops=dict(arrowstyle='-|>', color='k', lw=1.5, alpha=0.7))
    ax.text(3.5, 8.5, 'AK', fontsize=7, color='k', alpha=0.7)
    ax.annotate('', xy=(ra_x, ra_y), xytext=(rv_x, rv_y), arrowprops=dict(arrowstyle='-|>', color='k', lw=1.5, alpha=0.7))
    ax.text(6.5, 6.5, 'TK', fontsize=7, color='k', alpha=0.7)
    ax.annotate('', xy=(rv_x, rv_y), xytext=(8.5, 7), arrowprops=dict(arrowstyle='-|>', color='k', lw=1.5, alpha=0.7))
    ax.text(7.8, 6.2, 'PK', fontsize=7, color='k', alpha=0.7)

    # Vessels
    ax.annotate('', xy=(1, 8.5), xytext=(2.5, 8.5), arrowprops=dict(arrowstyle='->', color=arterial, lw=2))
    ax.annotate('', xy=(9, 8.5), xytext=(8.5, 8.5), arrowprops=dict(arrowstyle='->', color=venous, lw=2))
    ax.annotate('', xy=(ao_x, ao_y + ao_h), xytext=(ao_x, 12), arrowprops=dict(arrowstyle='->', color=arterial, lw=2))
    ax.annotate('', xy=(8.5, 7), xytext=(9.5, 7), arrowprops=dict(arrowstyle='->', color=venous, lw=2))

    # Info panel (proxy values from ODE state)
    HR = 60 / params['T_heart']
    info = (f"⏱ t = {t_cur:.2f} s\n"
            f"💓 V_LV = {V_lv:.1f} ml\n"
            f"📈 E_LV = {E_lv:.2f}\n"
            f"🔄 HR = {HR:.0f} bpm\n")
            # Note: SV, EF, CO are calculated later in calculate_hemodynamics
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Model Parameters Display
    param_str = (f"ODE Params:\n"
                 f"E_max: {params['Emax']}\n"
                 f"E_min: {params['Emin']}\n"
                 f"R_sys: {params['Ra']}")
    ax.text(0.98, 0.02, param_str, transform=ax.transAxes,
            fontsize=7, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))

def calculate_hemodynamics(V_lv, P_lv, t, params):
    """Calculate hemodynamic indices from simulation."""
    dt = t[1] - t[0] if len(t) > 1 else 0.002

    # Calculate EDV and ESV from the last full cardiac cycle
    cycle_length = int(params['T_heart'] / dt)
    num_cycles = int(len(t) / cycle_length) if cycle_length > 0 else 1
    if num_cycles == 0: num_cycles = 1

    start_of_last_cycle = (num_cycles - 1) * cycle_length
    if start_of_last_cycle < 0: start_of_last_cycle = 0
    end_of_last_cycle = min(num_cycles * cycle_length, len(V_lv))

    V_lv_last_cycle = V_lv[start_of_last_cycle:end_of_last_cycle]
    P_lv_last_cycle = P_lv[start_of_last_cycle:end_of_last_cycle]

    if len(V_lv_last_cycle) > 0:
        EDV = np.max(V_lv_last_cycle)
        ESV = np.min(V_lv_last_cycle)
    else:
        EDV = np.max(V_lv)
        ESV = np.min(V_lv)

    SV = EDV - ESV
    EF = (SV / EDV) * 100 if EDV > 0 else 0
    HR = 60 / params['T_heart']
    CO = (SV * HR) / 1000.0 # ml/beat * beat/min -> L/min

    # Stroke Work (SW) - Approximate area inside the loop
    # Use only the last cycle for more accurate calculation
    if len(V_lv_last_cycle) > 1:
        dV_cycle = np.diff(V_lv_last_cycle)
        P_avg_cycle = (P_lv_last_cycle[:-1] + P_lv_last_cycle[1:]) / 2
        SW = np.trapz(P_avg_cycle * dV_cycle) # mmHg * ml
    else:
        # Fallback if last cycle is too short
        dV = np.diff(V_lv)
        P_avg = (P_lv[:-1] + P_lv[1:]) / 2
        SW = np.trapz(P_avg * dV)

    # Retrieve stored values from params (calculated in plot_pv_loop)
    P_es = getattr(params, 'P_es', P_lv[np.argmin(V_lv)]) # Fallback if not stored
    ESV_calc = getattr(params, 'ESV', ESV)
    EDV_calc = getattr(params, 'EDV', EDV)

    # Recalculate Emax and Ea for hemodynamics dictionary
    Emax = P_es / (ESV_calc - params['Vd']) if ESV_calc > params['Vd'] else float('nan')
    Ea = P_es / (EDV_calc - ESV_calc) if EDV_calc != ESV_calc else float('nan')
    coupling = Emax / Ea if Ea > 0 and not np.isnan(Emax) else float('nan')

    return {
        'EDV': EDV_calc,
        'ESV': ESV_calc,
        'SV': SV,
        'EF': EF,
        'CO': CO,
        'SW': SW,
        'P_es': P_es, # Include for explanation figure
        'Emax': Emax, # Include for explanation figure
        'Ea': Ea,     # Include for explanation figure
        'coupling': coupling # Include for explanation figure
    }


# --- NEW: ENHANCED EXPLANATION FIGURE FUNCTION ---
def create_hemodynamics_explanation_figure(params, t, V_lv_sim, P_lv_sim, V_rv_sim, P_rv_sim,
                                           P_ao_sim, E_lv_sim, ecg, hemo):
    """Create a comprehensive figure explaining hemodynamics with chamber connections"""

    fig = plt.figure(figsize=(18, 12), facecolor='#f8f9fa')
    gs = fig.add_gridspec(3, 3, hspace=0.25, wspace=0.3,
                          height_ratios=[1.2, 1, 0.8],
                          top=0.95, bottom=0.05, left=0.05, right=0.95)

    # Color scheme
    colors = {
        'oxygenated': '#FF6B6B',  # Red/pink for oxygenated blood
        'deoxygenated': '#4ECDC4',  # Cyan/teal for deoxygenated blood
        'heart': '#2C3E50',
        'text': '#2C3E50',
        'ecg': '#2ECC71',
        'pv': '#E74C3C'
    }

    # ============================================================
    # PANEL 1: Heart Anatomy with Blood Flow (Top Left)
    # ============================================================
    ax_heart = fig.add_subplot(gs[0, 0])
    ax_heart.set_xlim(0, 12)
    ax_heart.set_ylim(0, 14)
    ax_heart.set_aspect('equal')
    ax_heart.axis('off')
    ax_heart.set_facecolor('#f8f9fa')
    ax_heart.set_title('🫀 Cardiac Anatomy & Blood Flow', fontsize=14, fontweight='bold', pad=15)

    # Left Atrium (LA) - Oxygenated blood
    la = Rectangle((2.5, 9.5), 2.0, 1.5,
                        fill=True, facecolor=colors['oxygenated'], alpha=0.7,
                        edgecolor='darkred', linewidth=2)
    ax_heart.add_patch(la)
    ax_heart.text(3.5, 10.5, 'LA', ha='center', va='center',
                 fontsize=12, fontweight='bold', color='white')
    ax_heart.text(3.5, 9.2, 'Left Atrium', ha='center', fontsize=8, style='italic')

    # Left Ventricle (LV)
    lv = Circle((3.5, 7.0), radius=1.2, # Using circle for simplicity in this context
                        facecolor=colors['oxygenated'], alpha=0.85,
                        edgecolor='darkred', linewidth=2.5)
    ax_heart.add_patch(lv)
    ax_heart.text(3.5, 7.0, 'LV', ha='center', va='center',
                 fontsize=14, fontweight='bold', color='white')
    ax_heart.text(3.5, 3.8, 'Left Ventricle', ha='center', fontsize=8, style='italic')

    # Right Atrium (RA) - Deoxygenated blood
    ra = Rectangle((7.5, 9.5), 2.0, 1.5,
                        fill=True, facecolor=colors['deoxygenated'], alpha=0.7,
                        edgecolor='darkblue', linewidth=2)
    ax_heart.add_patch(ra)
    ax_heart.text(8.5, 10.5, 'RA', ha='center', va='center',
                 fontsize=12, fontweight='bold', color='white')
    ax_heart.text(8.5, 9.2, 'Right Atrium', ha='center', fontsize=8, style='italic')

    # Right Ventricle (RV)
    rv_points = [(8.5, 7.0 - 1.0), (8.5 + 1.0, 7.0), (8.5, 7.0 + 1.0), (8.5 - 1.0, 7.0)] # Diamond shape
    rv = Polygon(rv_points,
                        facecolor=colors['deoxygenated'], alpha=0.85,
                        edgecolor='darkblue', linewidth=2.5)
    ax_heart.add_patch(rv)
    ax_heart.text(8.5, 7.0, 'RV', ha='center', va='center',
                 fontsize=14, fontweight='bold', color='white')
    ax_heart.text(8.5, 3.8, 'Right Ventricle', ha='center', fontsize=8, style='italic')

    # Aorta
    aorta = Rectangle((1.5, 11.5), 1.5, 2.0,
                           fill=True, facecolor=colors['oxygenated'], alpha=0.9,
                           edgecolor='darkred', linewidth=2)
    ax_heart.add_patch(aorta)
    ax_heart.text(2.25, 12.8, 'Ao', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='white')

    # Pulmonary Artery
    pa = Rectangle((9.0, 11.5), 1.5, 2.0,
                        fill=True, facecolor=colors['deoxygenated'], alpha=0.9,
                        edgecolor='darkblue', linewidth=2)
    ax_heart.add_patch(pa)
    ax_heart.text(9.75, 12.8, 'PA', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='white')

    # Valves and flow arrows
    # Mitral Valve (LA -> LV)
    ax_heart.annotate('', xy=(3.5, 9.5), xytext=(3.5, 8.2),
                     arrowprops=dict(arrowstyle='->', lw=2, color='darkred', alpha=0.8))
    ax_heart.text(4.2, 9.2, 'Mitral', fontsize=7, color='darkred', fontweight='bold')

    # Aortic Valve (LV -> Ao)
    ax_heart.annotate('', xy=(3.0, 11.5), xytext=(3.0, 8.2),
                     arrowprops=dict(arrowstyle='->', lw=2, color='darkred', alpha=0.8))
    ax_heart.text(1.8, 10.2, 'Aortic', fontsize=7, color='darkred', fontweight='bold')

    # Tricuspid Valve (RA -> RV)
    ax_heart.annotate('', xy=(8.5, 9.5), xytext=(8.5, 8.0),
                     arrowprops=dict(arrowstyle='->', lw=2, color='darkblue', alpha=0.8))
    ax_heart.text(9.2, 9.2, 'Tricuspid', fontsize=7, color='darkblue', fontweight='bold')

    # Pulmonary Valve (RV -> PA)
    ax_heart.annotate('', xy=(9.0, 9.0), xytext=(9.0, 11.5),
                     arrowprops=dict(arrowstyle='->', lw=2, color='darkblue', alpha=0.8))
    ax_heart.text(9.8, 10.2, 'Pulmonic', fontsize=7, color='darkblue', fontweight='bold')

    # Systemic circulation (Ao -> body -> RA)
    ax_heart.annotate('', xy=(10.0, 8.0), xytext=(12.5, 8.0),
                     arrowprops=dict(arrowstyle='->', lw=2, color='purple', alpha=0.6,
                                   connectionstyle="arc3,rad=0.2"))
    ax_heart.text(11.5, 8.5, 'Systemic\nCirculation', ha='center', fontsize=7, color='purple')

    # Pulmonary circulation (PA -> lungs -> LA)
    ax_heart.annotate('', xy=(1.5, 8.0), xytext=(0, 8.0),
                     arrowprops=dict(arrowstyle='->', lw=2, color='green', alpha=0.6,
                                   connectionstyle="arc3,rad=-0.2"))
    ax_heart.text(0.75, 8.5, 'Pulmonary\nCirculation', ha='center', fontsize=7, color='green')

    # Oxygenation labels
    ax_heart.text(0.5, 13.0, '🩸 Oxygenated', fontsize=7, color='darkred',
                 bbox=dict(boxstyle='round', facecolor=colors['oxygenated'], alpha=0.3))
    ax_heart.text(10.5, 13.0, '💙 Deoxygenated', fontsize=7, color='darkblue',
                 bbox=dict(boxstyle='round', facecolor=colors['deoxygenated'], alpha=0.3))

    # Real-time pressure/volume display
    mid_idx = len(t) // 2
    info_text = (f"Current: t={t[mid_idx]:.2f}s\n"
                 f"LV: {V_lv_sim[mid_idx]:.0f}mL | {P_lv_sim[mid_idx]:.0f}mmHg\n"
                 f"RV: {V_rv_sim[mid_idx]:.0f}mL | {P_rv_sim[mid_idx]:.0f}mmHg")
    ax_heart.text(0.02, 0.98, info_text, transform=ax_heart.transAxes,
                 fontsize=8, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # ============================================================
    # PANEL 2: ECG Waveform (Top Center)
    # ============================================================
    ax_ecg = fig.add_subplot(gs[0, 1])
    ax_ecg.plot(t, ecg, color=colors['ecg'], linewidth=1.5, label='ECG')
    ax_ecg.fill_between(t, ecg, 0, where=(ecg > 0), alpha=0.3, color=colors['ecg'])
    ax_ecg.set_xlim(0, 2.4)
    ax_ecg.set_ylim(-0.4, 1.4)
    ax_ecg.set_xlabel('Time (seconds)', fontsize=10, fontweight='bold')
    ax_ecg.set_ylabel('Amplitude (mV)', fontsize=10, fontweight='bold')
    ax_ecg.set_title('⚡ Electrocardiogram (ECG)', fontsize=14, fontweight='bold')
    ax_ecg.grid(True, alpha=0.3, linestyle='--')
    ax_ecg.axhline(0, color='black', linewidth=0.5)

    # Annotate ECG waves
    wave_annotations = [
        ('P', 0.15, 0.25, 'Atrial\ndepolarization'),
        ('QRS', 0.35, 0.9, 'Ventricular\ndepolarization'),
        ('T', 0.65, 0.4, 'Ventricular\nrepolarization')
    ]

    for wave, x_pos, y_pos, label in wave_annotations:
        ax_ecg.annotate(wave, xy=(x_pos, y_pos), xytext=(x_pos, y_pos + 0.3),
                       ha='center', fontsize=11, fontweight='bold', color='darkred',
                       arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
        ax_ecg.text(x_pos, y_pos + 0.45, label, ha='center', fontsize=7, style='italic')

    # Mark cardiac cycle phases
    phases = [
        (0.0, 0.1, 'Diastole\n(Filling)'),
        (0.1, 0.3, 'Atrial\nContraction'),
        (0.3, 0.45, 'Systole\n(Ejection)'),
        (0.45, 0.8, 'Diastole\n(Relaxation)')
    ]

    for start, end, label in phases:
        ax_ecg.axvspan(start, end, alpha=0.1, color='gray')
        if start + (end-start)/2 < 0.8:
            ax_ecg.text(start + (end-start)/2, -0.35, label, ha='center', fontsize=7, rotation=0)

    # ============================================================
    # PANEL 3: PV Loop (Top Right)
    # ============================================================
    ax_pv = fig.add_subplot(gs[0, 2])

    # Plot the actual PV loop from simulation
    ax_pv.plot(V_lv_sim, P_lv_sim, color=colors['pv'], linewidth=3, label='PV Loop (ODE)')
    ax_pv.fill_between(V_lv_sim, P_lv_sim, alpha=0.15, color=colors['pv'])

    # Mark key points (use the ones identified in plot_pv_loop or calculated in hemodynamics)
    # We'll use the values from the hemo dict which were calculated based on the last cycle
    points = [
        (hemo['EDV'], 8, f'EDV\n({hemo["EDV"]:.0f} mL)', 'blue'),
        (hemo['ESV'], hemo['P_es'], f'ESV\n({hemo["ESV"]:.0f} mL)', 'red'),
        (hemo['EDV'], hemo['P_es'], f'ESP\n({hemo["P_es"]:.0f} mmHg)', 'purple')
    ]

    for x, y, label, color in points:
        ax_pv.scatter(x, y, c=color, s=80, zorder=5, edgecolors='white', linewidth=2)
        ax_pv.annotate(label, xy=(x, y), xytext=(x+10, y+5),
                      ha='left', fontsize=8, fontweight='bold',
                      arrowprops=dict(arrowstyle='->', color=color, lw=1))

    # Emax and Ea lines
    if not np.isnan(hemo['Emax']) and hemo['ESV'] > params['Vd']:
        ax_pv.plot([params['Vd'], hemo['ESV']], [0, hemo['P_es']],
                  '--', color='orange', linewidth=2, label=f'Emax = {hemo["Emax"]:.2f}')
    if not np.isnan(hemo['Ea']) and (hemo['EDV'] - hemo['ESV']) > 1:
        ax_pv.plot([hemo['ESV'], hemo['EDV']], [hemo['P_es'], 0],
                  ':', color='purple', linewidth=2, label=f'Ea = {hemo["Ea"]:.2f}')

    ax_pv.set_xlabel('Left Ventricular Volume (mL)', fontsize=10, fontweight='bold')
    ax_pv.set_ylabel('Left Ventricular Pressure (mmHg)', fontsize=10, fontweight='bold')
    ax_pv.set_title('📊 Pressure-Volume (PV) Loop', fontsize=14, fontweight='bold')
    ax_pv.grid(True, alpha=0.3, linestyle='--')
    ax_pv.legend(loc='lower right', fontsize=8)
    ax_pv.set_xlim(min(V_lv_sim)*0.95, max(V_lv_sim)*1.05)
    ax_pv.set_ylim(min(P_lv_sim)*0.95, max(P_lv_sim)*1.05)

    # Add stroke work area
    ax_pv.text(hemo['EDV']-30, max(P_lv_sim)*0.1, f'Stroke Work\nArea = {hemo["SW"]:.0f} mmHg·mL',
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8), fontsize=8)

    # ============================================================
    # PANEL 5: Hemodynamic Parameters (Middle Left)
    # ============================================================
    ax_hemo = fig.add_subplot(gs[1, 0])
    ax_hemo.axis('off')
    ax_hemo.set_title('📈 Hemodynamic Parameters', fontsize=12, fontweight='bold', pad=10)

    # Create a table-like display
    param_data = [
        ('Heart Rate (HR)', f"{60/params['T_heart']:.0f}", 'bpm', '💓'),
        ('Stroke Volume (SV)', f"{hemo['SV']:.1f}", 'mL', '🩸'),
        ('Cardiac Output (CO)', f"{hemo['CO']:.2f}", 'L/min', '⚡'),
        ('Ejection Fraction (EF)', f"{hemo['EF']:.1f}", '%', '📊'),
        ('End-Diastolic Volume (EDV)', f"{hemo['EDV']:.0f}", 'mL', '📥'),
        ('End-Systolic Volume (ESV)', f"{hemo['ESV']:.0f}", 'mL', '📤'),
        ('Emax', f"{hemo['Emax']:.2f}" if not np.isnan(hemo['Emax']) else 'NaN', 'mmHg/mL', '📈'),
        ('Ea', f"{hemo['Ea']:.2f}" if not np.isnan(hemo['Ea']) else 'NaN', 'mmHg/mL', '🎯'),
        ('Ventricular-Arterial Coupling', f"{hemo['coupling']:.2f}" if not np.isnan(hemo['coupling']) else 'NaN', '', '🔗')
    ]

    y_start = 0.9
    for i, (param, value, unit, icon) in enumerate(param_data):
        y_pos = y_start - i * 0.07
        color = '#2C3E50'
        # Check coupling for status color
        if param.startswith('Ventricular'): # Identify the coupling row
            if not np.isnan(hemo['coupling']):
                if 1.0 <= hemo['coupling'] <= 2.5:
                    color = '#27AE60'
                else:
                    color = '#E74C3C'
            else:
                color = '#E74C3C' # NaN is also an issue

        ax_hemo.text(0.1, y_pos, f"{icon} {param}:", fontsize=9, fontweight='bold',
                    transform=ax_hemo.transAxes)
        ax_hemo.text(0.7, y_pos, f"{value} {unit}", fontsize=9, color=color,
                    transform=ax_hemo.transAxes, fontweight='bold')

    # Coupling interpretation
    coupling = hemo['coupling']
    if not np.isnan(coupling):
        if 1.0 <= coupling <= 2.5:
            status = "✓ Optimal coupling - Efficient energy transfer"
            status_color = '#27AE60'
        elif coupling < 1.0:
            status = "⚠ Poor coupling - Ventricular overload risk"
            status_color = '#E74C3C'
        else:
            status = "⚡ Hyperdynamic - Increased cardiac workload"
            status_color = '#F39C12'
    else:
        status = "⚠ Coupling could not be calculated (check Emax/Ea)"
        status_color = '#E74C3C'

    ax_hemo.text(0.1, 0.15, status, fontsize=9, color=status_color,
                transform=ax_hemo.transAxes, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # ============================================================
    # PANEL 6: Pressure & Volume Dynamics (Middle Right)
    # ============================================================
    ax_dynamics = fig.add_subplot(gs[1, 1:])

    # Twin axes for pressure and volume
    ax_vol2 = ax_dynamics
    ax_pres2 = ax_dynamics.twinx()

    # Plot with proper scaling
    ax_vol2.plot(t, V_lv_sim, 'b-', linewidth=2, label='LV Volume', alpha=0.8)
    ax_pres2.plot(t, P_lv_sim, 'r-', linewidth=2, label='LV Pressure', alpha=0.8)
    ax_pres2.plot(t, P_ao_sim, 'orange', linewidth=1.5, label='Aortic Pressure', alpha=0.6, linestyle='--')

    ax_vol2.set_xlabel('Time (seconds)', fontsize=10, fontweight='bold')
    ax_vol2.set_ylabel('Volume (mL)', color='blue', fontsize=10, fontweight='bold')
    ax_pres2.set_ylabel('Pressure (mmHg)', color='red', fontsize=10, fontweight='bold')
    ax_vol2.set_title('📉 Ventricular Dynamics Over Time', fontsize=12, fontweight='bold')
    ax_vol2.grid(True, alpha=0.3, linestyle='--')
    ax_vol2.set_xlim(0, 2.4)
    ax_vol2.set_ylim(min(V_lv_sim)*0.95, max(V_lv_sim)*1.05)
    ax_pres2.set_ylim(min(min(P_lv_sim), min(P_ao_sim))*0.95, max(max(P_lv_sim), max(P_ao_sim))*1.05)

    # Mark cardiac cycle phases on dynamics plot
    for start, end, label in phases[:3]:
        ax_vol2.axvspan(start, end, alpha=0.1, color='gray')

    # Combine legends
    lines1, labels1 = ax_vol2.get_legend_handles_labels()
    lines2, labels2 = ax_pres2.get_legend_handles_labels()
    ax_vol2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

    # ============================================================
    # PANEL 7: Clinical Interpretation (Bottom)
    # ============================================================
    ax_clinical = fig.add_subplot(gs[2, :])
    ax_clinical.axis('off')
    ax_clinical.set_title('🏥 Clinical Interpretation', fontsize=12, fontweight='bold', pad=10)

    # Create interpretation text based on parameters
    interpretations = []

    # EF interpretation
    if hemo['EF'] < 40:
        interpretations.append("⚠️ Reduced Ejection Fraction (<40%) - Suggests systolic dysfunction")
    elif hemo['EF'] < 50:
        interpretations.append("📊 Borderline Ejection Fraction - Monitor closely")
    else:
        interpretations.append("✓ Normal Ejection Fraction (>50%) - Preserved systolic function")

    # Coupling interpretation
    if not np.isnan(hemo['coupling']):
        if hemo['coupling'] < 1.0:
            interpretations.append("⚠️ Ventricular-arterial uncoupling - Consider afterload reduction")
        elif hemo['coupling'] > 2.5:
            interpretations.append("⚡ Hyperdynamic state - Rule out high output states")
        else:
            interpretations.append("✓ Optimal ventricular-arterial coupling")
    else:
        interpretations.append("⚠️ Coupling could not be calculated (Emax or Ea invalid)")

    # Cardiac output
    if hemo['CO'] < 4.0:
        interpretations.append("⚠️ Low cardiac output - Assess for heart failure")
    elif hemo['CO'] > 6.0:
        interpretations.append("⚡ High cardiac output - Consider anemia, hyperthyroidism")
    else:
        interpretations.append("✓ Normal cardiac output (4-6 L/min)")

    # Display interpretations in a nice format
    for i, text in enumerate(interpretations):
        ax_clinical.text(0.05, 0.7 - i*0.2, text, fontsize=10,
                        transform=ax_clinical.transAxes,
                        bbox=dict(boxstyle='round', facecolor='#ECF0F1', alpha=0.7))

    # Add recommendations
    ax_clinical.text(0.55, 0.7, "💡 Clinical Recommendations:", fontsize=10, fontweight='bold',
                    transform=ax_clinical.transAxes)

    recommendations = []
    if hemo['EF'] < 40:
        recommendations.append("• Consider echocardiography for structural assessment")
    if not np.isnan(hemo['coupling']) and hemo['coupling'] < 1.0:
        recommendations.append("• Optimize afterload management")
    if hemo['CO'] < 4.0:
        recommendations.append("• Assess for fluid responsiveness")

    for i, rec in enumerate(recommendations[:3]):
        ax_clinical.text(0.55, 0.55 - i*0.12, rec, fontsize=9,
                        transform=ax_clinical.transAxes)

    if not recommendations:
        ax_clinical.text(0.55, 0.55, "• Continue current management", fontsize=9,
                        transform=ax_clinical.transAxes)
        ax_clinical.text(0.55, 0.43, "• Monitor clinical status", fontsize=9,
                        transform=ax_clinical.transAxes)

    # Add explanatory note about cross-referencing
    note = ("📖 Cross-Referencing Guide:\n"
            "• ECG → PV Loop: Electrical activation (QRS) precedes mechanical contraction (PV loop)\n"
            "• PV Loop → Hemodynamics: Loop area = Stroke Work; Width = Stroke Volume\n"
            "• Dynamics → PV Loop: Volume/pressure traces correspond to points on PV loop")

    ax_clinical.text(0.05, 0.05, note, fontsize=8, transform=ax_clinical.transAxes,
                    style='italic', bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.8))

    plt.suptitle('Comprehensive Cardiac Hemodynamics Analysis (ODE Model)', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    return fig


# Integration function to be added to your main code
def update_plots_with_explanation(E_lv_max, E_rv_max, C_ao, R_systemic, V_lv_0, T_heart):
    """Enhanced update function with explanation figure"""

    # Gather current parameters based on ODE model structure
    params = base_params_ode.copy()
    params.update({
        'Emax': E_lv_max, 'Emin': E_rv_max, # Note: E_rv_max slider maps to Emin for ODE
        'Ca': C_ao, 'Ra': R_systemic,       # Note: R_systemic slider maps to Ra for ODE
        'Vd': V_lv_0, 'T_heart': T_heart,   # Note: V_lv_0 slider maps to Vd for ODE
    })

    try:
        validate_params(params)
    except ValueError as e:
        print(f"⚠️ {e}")
        return

    # Run simulation using the ODE model
    t, V_lv_sim, P_lv_sim, V_rv_sim, P_rv_sim, P_ao_sim, E_lv_sim, ecg, flow_mitral, flow_aortic = simulate_heart(
        params, duration=max(2.4, T_heart * 3), fs=250
    )

    # Calculate hemodynamics
    hemo = calculate_hemodynamics(V_lv_sim, P_lv_sim, t, params)

    # Create enhanced figure
    fig = create_hemodynamics_explanation_figure(params, t, V_lv_sim, P_lv_sim,
                                                 V_rv_sim, P_rv_sim, P_ao_sim,
                                                 E_lv_sim, ecg, hemo)

    plt.show()
    return fig


def update_plots(Emax, Emin, Ra, Ca, Vd, T_heart):
    """Update all plots based on parameters."""
    params = base_params_ode.copy()
    params.update({
        'Emax': Emax, 'Emin': Emin,
        'Ra': Ra, 'Ca': Ca,
        'Vd': Vd, 'T_heart': T_heart
    })

    try:
        validate_params(params)
    except ValueError as e:
        print(f"⚠️ {e}")
        return

    # Simulate with ODE model
    t, V_lv_sim, P_lv_sim, V_rv_sim, P_rv_sim, P_ao_sim, E_lv_sim, ecg, flow_mitral, flow_aortic = simulate_heart(
        params, duration=max(2.4, T_heart * 3), fs=250
    )

    hemo = calculate_hemodynamics(V_lv_sim, P_lv_sim, t, params)

    mid_idx = len(t) // 2
    t_current = t[mid_idx]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    fig.suptitle(f"🫀 ODE Heart Model | HR = {60/T_heart:.0f} bpm", fontsize=14, fontweight='bold', y=1.02)

    # 1. Heart Diagram
    draw_heart(axes[0, 0], params, t_current,
               V_lv_sim[mid_idx], P_lv_sim[mid_idx],
               V_rv_sim[mid_idx], P_rv_sim[mid_idx],
               P_ao_sim[mid_idx], E_lv_sim[mid_idx])

    # 2. ECG
    axes[0, 1].plot(t, ecg, 'g-', linewidth=0.8, label='ECG')
    axes[0, 1].set_xlim(t[mid_idx] - 1.5, t[mid_idx] + 1.5)
    axes[0, 1].set_ylim(-0.6, 1.6)
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Amplitude (mV)')
    axes[0, 1].set_title('⚡ Electrocardiogram')
    axes[0, 1].grid(True, alpha=0.3, linestyle='--')
    axes[0, 1].axhline(0, color='k', linewidth=0.4)
    axes[0, 1].legend(fontsize=8)

    # 3. PV Loop (Now uses ODE results)
    plot_pv_loop(axes[1, 0], V_lv_sim, P_lv_sim, t, params)

    # 4. Volume and Pressure Dynamics
    ax_vol = axes[1, 1]
    ax_pres = ax_vol.twinx()
    ax_vol.plot(t, V_lv_sim, 'b-', linewidth=1.2, label='V_LV (ml)', zorder=2)
    ax_pres.plot(t, P_lv_sim, 'r-', linewidth=1.2, label='P_LV (mmHg)', zorder=2)
    ax_vol.set_ylabel('Volume (ml)', color='b', fontweight='bold')
    ax_pres.set_ylabel('Pressure (mmHg)', color='r', fontweight='bold')
    ax_vol.set_title('📈 Dynamics: Volume & Pressure (ODE)')
    ax_vol.grid(True, alpha=0.3, linestyle='--')
    ax_vol.set_xlim(t[mid_idx] - 1.5, t[mid_idx] + 1.5)

    # Combined legend
    lines1, labels1 = ax_vol.get_legend_handles_labels()
    lines2, labels2 = ax_pres.get_legend_handles_labels()
    ax_vol.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

    # Metrics Panel
    metrics = (f"📊 HEMODYNAMICS (Last Cycle):\n"
               f"• EDV = {hemo['EDV']:.1f} ml\n"
               f"• ESV = {hemo['ESV']:.1f} ml\n"
               f"• SV = {hemo['SV']:.1f} ml\n"
               f"• EF = {hemo['EF']:.1f} %\n"
               f"• CO = {hemo['CO']:.2f} L/min\n"
               f"• SW = {hemo['SW']:.0f} mmHg.ml")
    axes[1, 1].text(0.98, 0.98, metrics, transform=ax_vol.transAxes,
                   fontsize=8, ha='right', va='top',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.show()
    return fig

def export_simulation_data(params, t, V_lv, P_lv, P_ao, ecg, filename='heart_ode_data.csv'):
    """Export simulation results to CSV."""
    df = pd.DataFrame({
        'time_s': t,
        'V_LV_ml': V_lv,
        'P_LV_mmHg': P_lv,
        'P_Ao_mmHg': P_ao,
        'ECG_mV': ecg
    })
    df.to_csv(filename, index=False)
    print(f"Data exported: {filename} ({len(df)} rows)")
    return df

def interactive_heart_model():
    """Create the interactive widget interface."""
    # Widgets for ODE-specific parameters
    Emax_w = widgets.FloatSlider(value=base_params_ode['Emax'], min=0.5, max=4.0, step=0.1, description='E_max', style={'description_width': 'initial'}, continuous_update=False)
    Emin_w = widgets.FloatSlider(value=base_params_ode['Emin'], min=0.01, max=0.5, step=0.01, description='E_min', style={'description_width': 'initial'}, continuous_update=False)
    Ra_w = widgets.FloatSlider(value=base_params_ode['Ra'], min=0.1, max=3.0, step=0.1, description='R_a (Aortic)', style={'description_width': 'initial'}, continuous_update=False)
    Ca_w = widgets.FloatSlider(value=base_params_ode['Ca'], min=0.1, max=2.0, step=0.1, description='C_a (Arterial)', style={'description_width': 'initial'}, continuous_update=False)
    Vd_w = widgets.FloatSlider(value=base_params_ode['Vd'], min=1.0, max=20.0, step=1.0, description='V_d (Unstressed)', style={'description_width': 'initial'}, continuous_update=False)
    T_heart_w = widgets.FloatSlider(value=base_params_ode['T_heart'], min=0.4, max=1.2, step=0.05, description='T_heart (s)', style={'description_width': 'initial'}, continuous_update=False)

    export_btn = widgets.Button(description="📥 Export CSV", button_style='success', tooltip='Save simulation data')
    explain_btn = widgets.Button(description="🔍 Show Full Analysis", button_style='info', tooltip='Show detailed hemodynamics explanation')
    out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '10px'})

    def on_export_clicked(b):
        with out:
            clear_output()
            params = base_params_ode.copy()
            params.update({
                'Emax': Emax_w.value, 'Emin': Emin_w.value,
                'Ra': Ra_w.value, 'T_heart': T_heart_w.value
            })
            t, V_lv, P_lv, *rest = simulate_heart(params, duration=T_heart_w.value * 3)
            df = export_simulation_data(params, t, V_lv, P_lv, rest[5], rest[7]) # Access P_ao and ecg correctly
            display(df.head(3))
            print(f"\n💡 Total rows: {len(df)} | File: heart_ode_data.csv")

    def on_explain_clicked(b):
        with out:
            clear_output(wait=True)
            # Call the new function with current widget values
            update_plots_with_explanation(Emax_w.value, Emin_w.value, Ca_w.value,
                                          Ra_w.value, Vd_w.value, T_heart_w.value)


    export_btn.on_click(on_export_clicked)
    explain_btn.on_click(on_explain_clicked)

    def on_params_change(*args):
        with out:
            clear_output(wait=True)
            update_plots(Emax_w.value, Emin_w.value, Ra_w.value, Ca_w.value, Vd_w.value, T_heart_w.value)

    for w in [Emax_w, Emin_w, Ra_w, Ca_w, Vd_w, T_heart_w]:
        w.observe(on_params_change, names='value')

    on_params_change() # Initial plot

    ui = widgets.VBox([
        widgets.HTML("<h3 style='color:#2c3e50'>ODE-Based Heart Model</h3>"),
        widgets.HTML("<p style='font-size:0.9em; color:#555'><i>Adjust sliders to change ODE parameters and observe the resulting PV loop and dynamics.</i></p>"),
        widgets.HBox([Emax_w, Emin_w]),
        widgets.HBox([Ra_w, Ca_w]),
        widgets.HBox([Vd_w, T_heart_w]),
        widgets.HBox([export_btn, explain_btn]), # Add the new button
        out
    ], layout=widgets.Layout(padding='10px'))

    display(ui)
    print("✅ Model loaded! Adjust parameters to see the PV loop generated by the ODE.")
    print("🔍 Use the 'Show Full Analysis' button to view the detailed hemodynamics explanation.")


# --- 10. RUN ---
if __name__ == "__main__":
    print("🚀 Launching ODE-based interactive heart model...")
    print("📋 Features:")
    print("   • ODE-based LV, Arterial, Venous dynamics")
    print("   • Real-time PV loop generation")
    print("   • Hemodynamic calculations (SV, EF, CO, Emax/Ea, SW)")
    print("   • Data export to CSV")
    print("   • Enhanced hemodynamics explanation figure")
    print("\n" + "─ " * 70 + "\n")
    interactive_heart_model()

🚀 Launching ODE-based interactive heart model...
📋 Features:
   • ODE-based LV, Arterial, Venous dynamics
   • Real-time PV loop generation
   • Hemodynamic calculations (SV, EF, CO, Emax/Ea, SW)
   • Data export to CSV
   • Enhanced hemodynamics explanation figure

─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ 



✅ Model loaded! Adjust parameters to see the PV loop generated by the ODE.
🔍 Use the 'Show Full Analysis' button to view the detailed hemodynamics explanation.
